# Cars85 Data Frames (dplyr) — Solution Notebook (R)

**Goal:** Clean, transform, and explore the 1985 Automobile dataset so you can decide which car to add to your collection. Full working solutions, alternate implementations, extra practice answers, and a parameterised simulation are provided below.

**How to use:** Run the cells in order. Compare your skeleton answers with the code shown here. The printed outputs you should expect (from a typical run) are shown after each major step.

## Analysis Flowchart

![Cars85 dplyr Workflow](cars85_dataframe_flowchart.png)

```mermaid
flowchart TD
    A[1 Load Libraries<br/>readr + dplyr] --> B[2 Load CSV<br/>cars85.csv]
    B --> C[3 Inspect<br/>head + summary]
    C --> D[4 Clean<br/>select -normalized_losses]
    D --> E[5 Rename<br/>symboling → risk_factor]
    E --> F[6 Feature Eng<br/>mutate mpg_diff]
    F --> G[7 Filter<br/>mpg_diff > 0]
    G --> H[8 Arrange<br/>desc mpg_diff / engine_size]
    H --> I[9 Subset by Make<br/>+ re-arrange]
    I --> J[10 Insights & Decision]
    J --> K[More Practice + Simulation]
```


## 0. Setup — Load Libraries

In [ ]:
library(readr)
library(dplyr)
# Expected: packages load quietly (message=FALSE can be used in Rmd)


## 1. Load the Data

In [ ]:
cars <- read_csv("data/cars85.csv")
# If the file is in the working directory instead:
# cars <- read_csv("cars85.csv")
cars
# Expected (first few rows shown for illustration):
# # A tibble: 205 × 26
#   symboling normalized_losses make        fuel_type aspiration ...
#       <dbl>             <dbl> <chr>       <chr>     <chr>      ...
# 1         3                NA alfa-romero gas       std        ...
# 2         3                NA alfa-romero gas       std        ...
# ...


## 2. Inspect the Data Frame

In [ ]:
head(cars)
summary(cars)

# Key observations you should notice:
# - 205 rows, 26 columns
# - normalized_losses has many NAs (≈41 missing)
# - price has 4 NAs
# - highway_mpg ranges roughly 16–54, mean ≈ 30.8
# - engine_size ranges 61–326, mean ≈ 127
# - Most frequent makes: toyota (32), nissan (18), mazda (17), honda (13) ...


## 3. Clean — Drop `normalized_losses`

In [ ]:
cars <- cars %>% select(-normalized_losses)
dim(cars)          # should be 205 × 25
colnames(cars)     # normalized_losses gone


## 4. View Column Names (already done above)

In [ ]:
colnames(cars)


## 5. Rename `symboling` → `risk_factor`

In [ ]:
cars <- cars %>% rename(risk_factor = symboling)
colnames(cars)   # first column should now be risk_factor


## 6. Define MPG Threshold & Create Difference Column

In [ ]:
mpg_threshold <- 30
cars <- cars %>% mutate(mpg_diff_from_threshold = highway_mpg - mpg_threshold)

cars %>%
  select(make, highway_mpg, mpg_diff_from_threshold, engine_size, price) %>%
  head(10)

# Expected: new column appears; negative values mean the car is below the threshold.


## 7. Filter Cars That Exceed the Threshold

In [ ]:
mpg_exceeds_threshold <- cars %>% filter(mpg_diff_from_threshold > 0)
nrow(mpg_exceeds_threshold)   # typically 98 cars

mpg_exceeds_threshold %>%
  select(make, highway_mpg, mpg_diff_from_threshold, engine_size, price) %>%
  head(8)


## 8. Arrange by MPG Advantage (Descending)

In [ ]:
mpg_exceeds_threshold <- mpg_exceeds_threshold %>%
  arrange(desc(mpg_diff_from_threshold))

mpg_exceeds_threshold %>%
  select(make, highway_mpg, mpg_diff_from_threshold, engine_size, price) %>%
  head(8)

# Expected top cars (illustrative):
#   make       highway_mpg  mpg_diff  engine_size  price
#   honda             54         24           92   6479
#   chevrolet         53         23           61   5151
#   nissan            50         20          103   7099
#   toyota            47         17          110   7788
#   ...


## 9. Order All Cars by Engine Size (Descending)

In [ ]:
ordered_by_engine_size <- cars %>% arrange(desc(engine_size))

ordered_by_engine_size %>%
  select(make, engine_size, highway_mpg, price, risk_factor) %>%
  head(8)

# Expected top (largest engines):
#   jaguar / mercedes-benz with engine_size 300+
#   These cars have low highway_mpg (≈16-19) and high prices.


## 10. Focus on a Chosen Make

In [ ]:
chosen_make <- "toyota"   # change freely

chosen_make_details <- cars %>%
  filter(make == chosen_make) %>%
  arrange(desc(engine_size))

chosen_make_details %>%
  select(make, engine_size, highway_mpg, city_mpg, price, risk_factor)

# For toyota (n=32): max engine_size = 171, mean highway_mpg ≈ 32.9
# Try "honda" (smaller engines, higher avg MPG ≈ 35.5) or "volkswagen".


## 11. Quick Value Metric

In [ ]:
cars <- cars %>% mutate(value_score = highway_mpg / (price / 1000))

cars %>%
  filter(mpg_diff_from_threshold > 0, !is.na(value_score)) %>%
  arrange(desc(value_score)) %>%
  select(make, highway_mpg, price, value_score, engine_size) %>%
  head(10)

# High value_score ≈ cheap + efficient cars (often small Honda/Chevrolet/Toyota).


---
## Alternate Code Paths (complete solutions)


### Alternate A — Base R equivalents

In [ ]:
# Start from a fresh load if desired
cars_base <- read_csv("data/cars85.csv")

# drop column
cars_base$normalized_losses <- NULL
# or: cars_base <- cars_base[, setdiff(names(cars_base), "normalized_losses")]

# rename
names(cars_base)[names(cars_base) == "symboling"] <- "risk_factor"

# filter + order
mpg_ex_base <- cars_base[cars_base$highway_mpg > 30, ]
mpg_ex_base <- mpg_ex_base[order(-mpg_ex_base$highway_mpg), ]

head(mpg_ex_base[, c("make", "highway_mpg", "engine_size", "price")])


### Alternate B — `transmute` for a slim analysis frame

In [ ]:
slim <- cars %>%
  transmute(
    make,
    risk_factor,
    highway_mpg,
    engine_size,
    price,
    mpg_diff = highway_mpg - 30,
    value_score = highway_mpg / (price / 1000)
  )
head(slim)
# Only the listed columns remain; original cars is unchanged.


### Alternate C — Complex filter with `|` and `!`

In [ ]:
special <- cars %>%
  filter( (make == "toyota" | highway_mpg > 40),
          body_style != "convertible" )
special %>% count(make, body_style)


---
## More Practice — Solutions


### Practice 1 — Price & Body Style Filter

In [ ]:
affordable <- cars %>%
  filter(price < 10000, body_style %in% c("sedan", "hatchback")) %>%
  arrange(price)

nrow(affordable)   # typically around 40–50 cars
head(affordable %>% select(make, body_style, price, highway_mpg, engine_size), 8)
# Cheapest cars are usually subcompact sedans/hatchbacks from Chevrolet, Honda, Toyota, etc.


### Practice 2 — Risk Factor Buckets

In [ ]:
cars <- cars %>%
  mutate(risk_label = case_when(
    risk_factor <= 0 ~ "safe",
    risk_factor == 1 ~ "neutral",
    risk_factor >= 2 ~ "risky",
    TRUE ~ NA_character_
  ))
table(cars$risk_label)
# or
cars %>% count(risk_label)


### Practice 3 — Top-N per Make

In [ ]:
top_toyota <- cars %>% filter(make == "toyota") %>% arrange(desc(highway_mpg)) %>% slice(1)
top_nissan <- cars %>% filter(make == "nissan") %>% arrange(desc(highway_mpg)) %>% slice(1)
top_mazda  <- cars %>% filter(make == "mazda")  %>% arrange(desc(highway_mpg)) %>% slice(1)

bind_rows(top_toyota, top_nissan, top_mazda) %>%
  select(make, highway_mpg, engine_size, price, risk_factor)


---
## Simulation Section — Parameterised Pipeline & Sensitivity


In [ ]:
# --- EDITABLE PARAMETERS ---
sim_mpg_threshold <- 30
sim_chosen_make   <- "honda"
sim_max_price     <- 12000
sim_min_engine    <- 90
# ---------------------------

sim_result <- cars %>%
  mutate(mpg_diff = highway_mpg - sim_mpg_threshold) %>%
  filter(
    mpg_diff > 0,
    (make == sim_chosen_make | sim_chosen_make == "ANY"),
    (is.na(price) | price <= sim_max_price),
    engine_size >= sim_min_engine
  ) %>%
  arrange(desc(mpg_diff), desc(highway_mpg)) %>%
  select(make, body_style, highway_mpg, mpg_diff, engine_size, price, risk_factor)

cat("Parameters: threshold =", sim_mpg_threshold,
    "| make =", sim_chosen_make,
    "| max price =", sim_max_price,
    "| min engine =", sim_min_engine, "\n")
cat("Cars that survive the filters:", nrow(sim_result), "\n\n")
print(head(sim_result, 10))


### Sensitivity Table — Count of Cars Above Different Thresholds

In [ ]:
thresholds <- c(20, 25, 28, 30, 32, 35, 40, 45, 50)
counts <- sapply(thresholds, function(th) sum(cars$highway_mpg > th, na.rm = TRUE))
sens_df <- data.frame(threshold = thresholds, n_cars = counts)
print(sens_df)

# Optional visualisation (base R)
barplot(counts, names.arg = thresholds,
        main = "Cars with highway_mpg > threshold",
        xlab = "Highway MPG Threshold", ylab = "Number of cars",
        col = "steelblue", border = NA)


### Monte-Carlo Noise on Highway MPG

In [ ]:
set.seed(42)
n_sims <- 5
for (i in 1:n_sims) {
  noisy <- cars %>%
    mutate(hwy_noisy = highway_mpg + runif(n(), -2, 2),
           diff_noisy = hwy_noisy - 30) %>%
    filter(diff_noisy > 0) %>%
    arrange(desc(diff_noisy)) %>%
    select(make, highway_mpg, hwy_noisy, engine_size, price) %>%
    head(5)
  cat("=== Simulation", i, "===\n")
  print(noisy)
  cat("\n")
}
# Ranking of the very top cars is fairly stable; mid-pack can swap places under ±2 mpg noise.


## Reflection & Decision (example)

1. **Recommended short-list:** A high-MPG Honda (or Chevrolet) that still has a usable engine size and a low price (value_score leaders). If the collector prioritises engine character over pure efficiency, a mid-size Toyota with engine_size ≈ 150–170 and still > 30 highway mpg is attractive.

2. **Sensitivity:** Raising the threshold from 30 → 35 roughly halves the candidate pool; many large-engine cars disappear. Changing `chosen_make` completely changes the short-list, confirming that make preference is a strong prior.

3. **Missing data that would help:** reliability / TCO (total cost of ownership), parts availability in 2020s, actual collected-car market values, and a subjective “fun-to-drive” rating.

You now have a complete, reproducible dplyr pipeline for the 1985 cars data together with robustness checks. Excellent work!
